# Postcard migration

Tracker row **#16 (Postcard)** — legacy Strapi `postcards` → new `postcards`,
plus **tags → `facet_assignments`** (`owned_type='postcard'`) via the
Experience facet from the tags migration.

> **`postcards` has two sources.** Since 2026-08-11 the directory/album
> migration also writes to `postcards`: Restaurants/Events/Shopping albums have
> no Collection layer, so they land there directly. This notebook **adds** the
> legacy postcards around them and never overwrites them — their slugs are
> reserved before any slug is generated here.

Scope decisions (2026-08-07, album split added 2026-08-11):
- `album` → `collection_id` via the per-env `legacy_album_id_map` file;
  `collection_type_id` copied from that collection.
- If the album is in `legacy_album_postcard_id_map` instead (Restaurants/
  Events/Shopping — the album *is* a postcard), the legacy postcard gets
  `collection_id = NULL` and inherits `collection_type_id` + geo from that
  album-derived postcard. 0 such postcards in prod, 6 in dev.
- Postcards whose album is in **neither** map (Designer Tours) are **skipped**
  — they belong to the dx-card / Destination Expert migration (#11/#13).
  Postcards with **no album** migrate with `collection_id = NULL`, defaulted to
  Properties.
- `collection_id` is kept for every postcard whose album became a collection —
  the linkage is preserved rather than thrown away; geo is ALSO set directly.
- geo: legacy postcard has only `country`. Resolved country = postcard's
  country (by name, how geo migrated) else the parent's. `region_id` /
  `locality_id` inherit from the parent only when the parent's country matches
  the resolved country. `city_id` stays NULL.
- `status`: legacy has none — `isComplete` → `live`, else `draft`.
  `published_at` = legacy `createdAt` for live postcards.
- `slug`: ~10% empty in legacy → generated from name, de-duplicated in-run
  (id-sorted, so `foo-2` suffixes stay stable across re-runs).
- `tags` → `facet_assignments` via the per-env `legacy_tag_id_map` file.
- Dropped: `articleURL` (empty everywhere), `isFounderStory` (no v2 home),
  `album_themes` (empty on postcards), timestamps (except `published_at`).
  Deferred: `bookmarks` (#18), `memories` (#19), `property_itineraries` (#31).
- `location` / `event_details` / `website` stay NULL for legacy postcards — no
  legacy source. (Album-derived rows do populate them; the `ON CONFLICT` clause
  here leaves those columns untouched.)
- Writes `legacy_postcard_id_map_dev/_prod.json` for bookmarks/memories.

Prerequisites: geo, media, company, users, directory/album and **tags facet**
migrations done (i.e. `python scripts/migrate_data.py` ran clean).

Run cells top to bottom. Idempotent — safe to re-run.


In [ ]:
import os, re, json
from pathlib import Path

import requests
import psycopg
from dotenv import load_dotenv

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
load_dotenv(ROOT / ".env")

CMS_BASE_URL = os.environ["CMS_BASE_URL"].rstrip("/")
HEADERS = {"Authorization": f"Bearer {os.environ['CMS_API_TOKEN']}"}
DATABASE_URL = os.environ["DATABASE_URL"]
ENV_SUFFIX = {"development": "_dev", "production": "_prod"}.get(DATABASE_URL.rsplit("/", 1)[-1], "")


def slugify(text):
    return re.sub(r"[^a-z0-9]+", "-", (text or "").lower()).strip("-") or None


def attrs(item):
    """Entry fields — Strapi v4 nests them under 'attributes', v5 is flat."""
    return item.get("attributes", item)


def rel(obj):
    """Unwrap a populated relation — v4: {'data': {'attributes': {...}}}, v5: flat dict."""
    if isinstance(obj, dict) and "data" in obj:
        obj = obj["data"]
    if not obj:
        return None
    return obj.get("attributes", obj)


def rel_many(obj):
    """Unwrap a populated to-many relation into a list of flat dicts."""
    if isinstance(obj, dict) and "data" in obj:
        obj = obj["data"]
    return [attrs(x) for x in (obj or [])]


def fetch_all(path, params=None):
    """Fetch every page of a Strapi collection endpoint (data/meta envelope)."""
    items, page = [], 1
    while True:
        p = {"pagination[page]": page, "pagination[pageSize]": 100, "sort": "id", **(params or {})}
        r = requests.get(f"{CMS_BASE_URL}{path}", headers=HEADERS, params=p, timeout=120)
        r.raise_for_status()
        body = r.json()
        items.extend(body["data"])
        pg = body.get("meta", {}).get("pagination", {})
        if page >= pg.get("pageCount", 1):
            return items
        page += 1


conn = psycopg.connect(DATABASE_URL)
print("connected to:", DATABASE_URL.rsplit("/", 1)[-1])


def load_map(name, required=True):
    path = ROOT / f"{name}{ENV_SUFFIX}.json"
    if not required and not path.exists():
        return {}
    return {int(k): int(v) for k, v in json.loads(path.read_text()).items()}


# per-environment maps from the album and tag migrations
album_map = load_map("legacy_album_id_map")                            # album -> collection
album_postcard_map = load_map("legacy_album_postcard_id_map", False)   # album -> postcard
tag_map = load_map("legacy_tag_id_map")
print(f"loaded {len(album_map)} album->collection mappings, "
      f"{len(album_postcard_map)} album->postcard mappings, "
      f"{len(tag_map)} tag mappings ({ENV_SUFFIX or 'no suffix'})")


connected to: development
loaded 1261 album mappings, 701 tag mappings (_dev)


## 1. Fetch all postcards (~30 paginated requests, expect 2924)

In [2]:
postcards = sorted(fetch_all("/api/postcards", {"populate": "*"}), key=lambda x: x["id"])
print(f"fetched {len(postcards)} postcards")

fetched 2924 postcards


## 2. DB lookup maps + media find-or-create helper

Collections give `collection_type_id` + inherited geo. Album-derived postcards
(Restaurants/Events/Shopping) do the same job for legacy postcards hanging off
a non-dedicated album — and their slugs are **reserved**, so the upsert in
section 3 can never land on one of those rows. Countries are matched by name
(how geo migrated). Media is keyed by normalized url (same as
`scripts/media.py` — rows are reused, never duplicated).


In [ ]:
conn.rollback()  # clear any aborted transaction from a previous failed run

with conn.cursor() as cur:
    cur.execute("SELECT id, collection_type_id, country_id, region_id, locality_id FROM collections")
    coll_info = {i: (ct, co, rg, lo) for i, ct, co, rg, lo in cur.fetchall()}
    cur.execute("SELECT LOWER(name), id FROM countries")
    country_by_name = dict(cur.fetchall())
    cur.execute("SELECT slug, id FROM collection_types")
    ct_id_by_slug = dict(cur.fetchall())
    cur.execute("SELECT url, id FROM media")
    media_by_url = dict(cur.fetchall())
    cur.execute(
        "SELECT id, collection_type_id, country_id, region_id, locality_id, slug "
        "FROM postcards WHERE id = ANY(%s)", (list(album_postcard_map.values()),))
    album_pc_rows = cur.fetchall()

pc_info = {i: (ct, co, rg, lo) for i, ct, co, rg, lo, _ in album_pc_rows}
reserved_slugs = {s for *_, s in album_pc_rows}

DEFAULT_CT_ID = ct_id_by_slug["properties"]  # fallback for postcards with no album
print(f"lookups: {len(coll_info)} collections, {len(country_by_name)} countries, "
      f"{len(ct_id_by_slug)} collection_types, {len(media_by_url)} media, "
      f"{len(pc_info)} album-derived postcards (slugs reserved)")


def media_id_for(image, cur):
    """Find-or-create a media row for a populated Strapi file."""
    if not image or not image.get("url"):
        return None
    url = image["url"].strip()
    if url.startswith("/"):
        url = CMS_BASE_URL + url
    if url in media_by_url:
        return media_by_url[url]
    cur.execute(
        "INSERT INTO media (url, mime_type, alt, width, height) VALUES (%s, %s, %s, %s, %s) RETURNING id",
        (url, image.get("mime"), image.get("alternativeText") or image.get("name"),
         image.get("width"), image.get("height")),
    )
    media_by_url[url] = cur.fetchone()[0]
    return media_by_url[url]


lookups: 1261 collections, 268 countries, 5 collection_types, 9743 media


## 3. Postcard → `postcards`

Upsert on `slug`, with album-derived slugs reserved up front. How `album`
resolves:

| The postcard's album… | `collection_id` | `collection_type_id` | geo from |
|---|---|---|---|
| became a **collection** (Properties) | that collection | copied from it | that collection |
| became a **postcard** (Restaurants/Events/Shopping) | **NULL** | from the album-derived postcard | that postcard |
| is **Designer Tours** (neither map) | **skipped** → dx-card migration | | |
| is **missing** | NULL | Properties (default) | — |


In [ ]:
conn.rollback()

# album-derived postcards (Restaurants/Events/Shopping) already own their slugs
# — reserve them so an upsert here can never overwrite those rows
used_slugs = set(reserved_slugs)

def unique_slug(base):
    base = base or "postcard"
    slug, n = base, 2
    while slug in used_slugs:
        slug = f"{base}-{n}"
        n += 1
    used_slugs.add(slug)
    return slug

postcard_map = {}   # legacy postcard id -> new postcard id

skipped_no_name, skipped_unmigrated_album, no_album = [], [], []
album_is_postcard, missing_country = [], []

with conn.cursor() as cur:
    for pc in postcards:
        a = attrs(pc)
        name = (a.get("name") or "").strip()
        if not name:
            skipped_no_name.append(pc["id"])
            continue

        # album -> collection (+ its type and geo); an album of a non-dedicated
        # type is itself a postcard -> no collection to point at
        album = rel(a.get("album"))
        collection_id = ct_id = None
        c_country = c_region = c_locality = None
        if album:
            collection_id = album_map.get(album["id"])
            if collection_id:
                ct_id, c_country, c_region, c_locality = coll_info[collection_id]
            elif album["id"] in album_postcard_map:
                parent_pc_id = album_postcard_map[album["id"]]
                ct_id, c_country, c_region, c_locality = pc_info[parent_pc_id]
                album_is_postcard.append((pc["id"], name, album.get("name")))
            else:  # Designer Tours album -> dx-card migration later
                skipped_unmigrated_album.append((pc["id"], name, album.get("name")))
                continue
        else:
            no_album.append((pc["id"], name))
            ct_id = DEFAULT_CT_ID

        # geo: postcard country (by name) wins, else the parent's; region/locality
        # inherit from the parent only when its country matches
        country = rel(a.get("country"))
        country_id = country_by_name.get((country.get("name") or "").strip().lower()) if country else None
        if country and not country_id:
            missing_country.append((pc["id"], country.get("name")))
        country_id = country_id or c_country
        region_id = c_region if (country_id and country_id == c_country) else None
        locality_id = c_locality if (country_id and country_id == c_country) else None

        slug = unique_slug((a.get("slug") or "").strip() or slugify(name))
        cover_id = media_id_for(rel(a.get("coverImage")), cur)
        status = "live" if a.get("isComplete") else "draft"
        published_at = a.get("createdAt") if status == "live" else None

        cur.execute(
            """
            INSERT INTO postcards
                (name, intro, slug, story, collection_type_id, collection_id,
                 country_id, region_id, city_id, locality_id, copyright,
                 is_featured, priority, cover_media_id, status, published_at)
            VALUES (%s, %s, %s, %s, %s, %s, %s, %s, NULL, %s, %s, %s, %s, %s, %s, %s)
            ON CONFLICT (slug) DO UPDATE
            SET name = EXCLUDED.name,
                intro = EXCLUDED.intro,
                story = EXCLUDED.story,
                collection_type_id = EXCLUDED.collection_type_id,
                collection_id = EXCLUDED.collection_id,
                country_id = EXCLUDED.country_id,
                region_id = EXCLUDED.region_id,
                locality_id = EXCLUDED.locality_id,
                copyright = EXCLUDED.copyright,
                is_featured = EXCLUDED.is_featured,
                priority = EXCLUDED.priority,
                cover_media_id = EXCLUDED.cover_media_id,
                status = EXCLUDED.status,
                published_at = EXCLUDED.published_at
            RETURNING id
            """,
            (name,
             (a.get("intro") or "").strip() or None,
             slug,
             (a.get("story") or "").strip() or None,
             ct_id, collection_id,
             country_id, region_id, locality_id,
             (a.get("copyright") or "").strip() or None,
             bool(a.get("isFeatured")), a.get("priority") or 0,
             cover_id, status, published_at),
        )
        postcard_map[pc["id"]] = cur.fetchone()[0]

conn.commit()
print(f"postcards upserted: {len(postcard_map)}")
print(f"skipped (no name): {skipped_no_name}")
print(f"skipped, album not migrated = Designer Tours ({len(skipped_unmigrated_album)}): {skipped_unmigrated_album[:10]}")
print(f"no album -> defaulted to Properties, no collection ({len(no_album)}): {no_album[:20]}")
print(f"album is itself a postcard (non-dedicated type) -> collection_id NULL, "
      f"type/geo inherited ({len(album_is_postcard)}): {album_is_postcard[:20]}")
print(f"MANUAL REVIEW country not found ({len(missing_country)}): {missing_country[:20]}")


postcards upserted: 2422
skipped (no name): [995, 1005, 1006, 1007, 1025, 1027, 1028, 1029, 1030, 1031, 1034, 1056, 1064, 1065, 1066, 1067, 1069, 1073, 1074, 1077, 1080, 1105, 1106, 1118, 1126, 1127, 1135, 1136, 1137, 1138, 1139, 1164, 1165, 1166, 1167, 1168, 1169, 1170, 1171, 1172, 1173, 1174, 1175, 1176, 1177, 1178, 1180, 1210, 1211, 1239, 1241, 1248, 1251, 1266, 1267, 1268, 1269, 1270, 1271, 1272, 1273, 1274, 1275, 1276, 1278, 1280, 1281, 1286, 1289, 1327, 1328, 1329, 1330, 1331, 1332, 1333, 1334, 1335, 1421, 1422, 1442, 1443, 1453, 1466, 1467, 1468, 1480, 1481, 1483, 1484, 1487, 1488, 1489, 1490, 1491, 1492, 1493, 1494, 1495, 1496, 1497, 1498, 1499, 1544, 1545, 1546, 1558, 1559, 1560, 1561, 1562, 1567, 1569, 1571, 1572, 1577, 1579, 1586, 1587, 1595, 1596, 1597, 1598, 1628, 1629, 1630, 1631, 1638, 1639, 1640, 1659, 1660, 1673, 1675, 1676, 1677, 1678, 1679, 1738, 2236, 2237, 2298, 2299, 2300, 2301, 4437]
skipped, album not migrated = Designer Tours (356): [(1, 'LUM - the Place of Mem

## 4. tags → `facet_assignments` (owned_type = postcard)

Each legacy postcard↔tag link becomes one assignment row pointing at the
Experience facet_value from the tags migration. Duplicate-tag merges land on
the same value; the unique key de-duplicates. Idempotent
(`ON CONFLICT DO NOTHING`).

In [5]:
conn.rollback()

assignments = 0
unmapped_tags = []
with conn.cursor() as cur:
    for pc in postcards:
        new_id = postcard_map.get(pc["id"])
        if not new_id:
            continue
        for t in rel_many(attrs(pc).get("tags")):
            fv_id = tag_map.get(t["id"])
            if not fv_id:
                unmapped_tags.append((pc["id"], t["id"], t.get("name")))
                continue
            cur.execute(
                """
                INSERT INTO facet_assignments (owned_type, owned_id, facet_value_id)
                VALUES ('postcard', %s, %s)
                ON CONFLICT (owned_type, owned_id, facet_value_id) DO NOTHING
                """,
                (new_id, fv_id),
            )
            assignments += cur.rowcount

conn.commit()
print(f"facet_assignments inserted this run: {assignments}")
print(f"MANUAL REVIEW legacy tags not in map ({len(unmapped_tags)}): {unmapped_tags[:20]}")

facet_assignments inserted this run: 3050
MANUAL REVIEW legacy tags not in map (0): []


## 5. Save the legacy postcard id map

`legacy_postcard_id_map_dev/_prod.json` (legacy postcard id → new id) — the
bookmarks (#18) and memories (#19) migrations need it.

In [6]:
out = ROOT / f"legacy_postcard_id_map{ENV_SUFFIX}.json"
out.write_text(json.dumps({str(k): str(v) for k, v in postcard_map.items()}, indent=2))
print(f"saved {len(postcard_map)} legacy->new postcard id mappings to {out}")

saved 2422 legacy->new postcard id mappings to c:\Users\ReTechie\Desktop\postcard\postcard-migration\legacy_postcard_id_map_dev.json


## 6. OPTIONAL — author circles

Legacy `postcard.user` → Circle `author` (`owned_type = 'postcard'`), via the
per-env legacy user id map. Skip this cell if circles should wait.

In [7]:
conn.rollback()

user_map_file = ROOT / f"legacy_user_id_map{ENV_SUFFIX}.json"
user_map = {int(k): int(v) for k, v in json.loads(user_map_file.read_text()).items()}
print(f"loaded {len(user_map)} user mappings from {user_map_file.name}")

author_rows = 0
unmapped_users = []
with conn.cursor() as cur:
    for pc in postcards:
        new_id = postcard_map.get(pc["id"])
        if not new_id:
            continue
        u = rel(attrs(pc).get("user"))
        if not u:
            continue
        new_uid = user_map.get(u["id"])
        if not new_uid:
            unmapped_users.append((pc["id"], u["id"]))
            continue
        cur.execute(
            """
            INSERT INTO circles (user_id, owned_type, owned_id, relationship)
            VALUES (%s, 'postcard', %s, 'author')
            ON CONFLICT (user_id, owned_type, owned_id, relationship) DO NOTHING
            """,
            (new_uid, new_id),
        )
        author_rows += cur.rowcount

conn.commit()
print(f"author circles inserted this run: {author_rows}")
print(f"MANUAL REVIEW legacy users not in map ({len(unmapped_users)}): {unmapped_users[:20]}")

loaded 3279 user mappings from legacy_user_id_map_dev.json
author circles inserted this run: 1334
MANUAL REVIEW legacy users not in map (0): []


## 7. Verification

Two invariants after both step 5 and step 7 have run:

- **`bad: nonded w/ coll` must be 0** — no postcard of a non-dedicated type may
  carry a `collection_id`.
- Restaurants/Events/Shopping totals = the album-derived counts (341 / 91 / 235
  in prod) plus any legacy postcards under those albums (0 in prod).

Expected totals (prod, 2026-08-11): ~6,600+ Properties postcards from this step
plus the 667 album-derived rows; most with a collection and cover; assignments
≈ sum of legacy postcard-tag links; 0 duplicate slugs.


In [ ]:
with conn.cursor() as cur:
    print(f"{'collection type':22} {'total':>7} {'w/ coll':>8} {'no coll':>8}")
    cur.execute("""
        SELECT ct.name, COUNT(p.id), COUNT(p.collection_id),
               COUNT(p.id) - COUNT(p.collection_id)
        FROM collection_types ct
        LEFT JOIN postcards p ON p.collection_type_id = ct.id
        GROUP BY ct.id, ct.name ORDER BY MIN(ct.priority)
    """)
    for name, n, with_coll, without in cur.fetchall():
        print(f"{name:22} {n:7} {with_coll:8} {without:8}")

    for label, q in [
        ("postcards total",     "SELECT COUNT(*) FROM postcards"),
        ("with collection",     "SELECT COUNT(*) FROM postcards WHERE collection_id IS NOT NULL"),
        ("album-derived",       "SELECT COUNT(*) FROM postcards WHERE website IS NOT NULL OR event_details IS NOT NULL"),
        ("bad: nonded w/ coll", """
            SELECT COUNT(*) FROM postcards p JOIN collection_types ct ON ct.id = p.collection_type_id
            WHERE ct.has_dedicated_collection = false AND p.collection_id IS NOT NULL"""),
        ("with cover media",    "SELECT COUNT(*) FROM postcards WHERE cover_media_id IS NOT NULL"),
        ("with country",        "SELECT COUNT(*) FROM postcards WHERE country_id IS NOT NULL"),
        ("status = live",       "SELECT COUNT(*) FROM postcards WHERE status = 'live'"),
        ("facet assignments",   "SELECT COUNT(*) FROM facet_assignments WHERE owned_type = 'postcard'"),
        ("postcards w/ facets", "SELECT COUNT(DISTINCT owned_id) FROM facet_assignments WHERE owned_type = 'postcard'"),
        ("author circles",      "SELECT COUNT(*) FROM circles WHERE owned_type = 'postcard' AND relationship = 'author'"),
        ("dup slugs (want 0)",  "SELECT COUNT(*) FROM (SELECT slug FROM postcards GROUP BY slug HAVING COUNT(*) > 1) d"),
    ]:
        cur.execute(q)
        print(f"{label:20}: {cur.fetchone()[0]}")
conn.close()


Properties            : 2416
Restaurants           : 6
Events                : 0
Shopping              : 0
Destination Expert    : 0
postcards total     : 2422
with collection     : 2340
with cover media    : 2349
with country        : 2414
status = live       : 2252
facet assignments   : 3050
postcards w/ facets : 2265
author circles      : 1334
dup slugs (want 0)  : 0
